# Day 8: Scaled Dot-Product Attention

## Core Theory (Just-in-Time)

Scaled Dot-Product Attention is the foundational mechanism behind the Transformer architecture. Instead of processing tokens sequentially like RNNs, Transformers process all tokens simultaneously using an attention matrix that defines how much each token should "attend" to every other token.

Given a Query matrix $Q$, Key matrix $K$, and Value matrix $V$, the attention output is computed as:

$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

**Why scale by $\sqrt{d_k}$?** 
As the dimensionality of the key vectors ($d_k$) grows, the dot products $QK^T$ can become extremely large in magnitude. Large values push the softmax function into regions with near-zero gradients (saturating the softmax), making backpropagation incredibly slow or halting it completely. Scaling down the dot products by $\sqrt{d_k}$ ensures the variance remains close to 1 (assuming $Q$ and $K$ have unit variance), keeping gradients stable.


In [1]:
import numpy as np

def scaled_dot_product_attention(
    q: np.ndarray, 
    k: np.ndarray, 
    v: np.ndarray, 
    mask: np.ndarray | None = None
) -> np.ndarray:
    """
    Computes Scaled Dot-Product Attention.
    
    Args:
        q: Query matrix of shape (..., seq_len_q, d_k).
        k: Key matrix of shape (..., seq_len_k, d_k).
        v: Value matrix of shape (..., seq_len_v, d_v).
        mask: Optional boolean/binary mask matrix of shape (..., seq_len_q, seq_len_k) 
              to be broadcasted. Elements where mask evaluates to True or 1 will be ignored.
              
    Returns:
        np.ndarray: The output context vectors of shape (..., seq_len_q, d_v).
    """
    d_k = q.shape[-1]
    
    # Compute dot products QK^T
    # q is (..., seq_len_q, d_k)
    # k is (..., seq_len_k, d_k) -> transposed to (..., d_k, seq_len_k)
    # scores is (..., seq_len_q, seq_len_k)
    scores = np.matmul(q, np.swapaxes(k, -2, -1))
    
    # Scale by sqrt(d_k)
    scores = scores / np.sqrt(d_k)
    
    # Apply mask if provided
    if mask is not None:
        # We add a large negative number instead of -inf to avoid NaNs if a whole row is masked
        scores = np.where(mask, -1e9, scores)
        
    # Softmax for attention weights
    # Subtract max for numerical stability (doesn't change mathematical softmax output)
    scores_max = np.max(scores, axis=-1, keepdims=True)
    exp_scores = np.exp(scores - scores_max)
    weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    
    # Multiply by values V
    # weights is (..., seq_len_q, seq_len_k)
    # v is (..., seq_len_k, d_v)
    # output is (..., seq_len_q, d_v)
    output = np.matmul(weights, v)
    
    return output

# Demonstration:
np.random.seed(42)
batch_size = 2
seq_len = 3
d_k = 4
d_v = 4

q_sample = np.random.randn(batch_size, seq_len, d_k)
k_sample = np.random.randn(batch_size, seq_len, d_k)
v_sample = np.random.randn(batch_size, seq_len, d_v)

attention_output = scaled_dot_product_attention(q_sample, k_sample, v_sample)
print("Output shape:", attention_output.shape)
print("Output sample:\n", attention_output[0])


Output shape: (2, 3, 4)
Output sample:
 [[-0.50956193  0.05588786  0.81102648  0.69426007]
 [-0.59107637 -0.1303306   0.63239038  0.75161766]
 [-0.24829184 -0.74099788  0.51188314  0.33323852]]


## Common Pitfalls in Production

1.  **Missing the Scaling Factor:** Failing to divide by $\sqrt{d_k}$ leads to unstable gradients and training stagnation.
2.  **$O(N^2)$ Memory Complexity:** Standard attention computes an $N \times N$ matrix, making it computationally and memory expensive for long sequences. Production models often use FlashAttention to optimize this.
3.  **Numerical Instability in Softmax:** Exponentiating large positive values leads to overflow. Always subtract the maximum value along the axis before computing `np.exp`, which mathematically preserves the Softmax but ensures computational stability.
4.  **Incorrect Masking Values:** Adding `-inf` to masked positions works mathematically but can lead to `NaN` errors during backpropagation if a row is completely masked. Often, `-1e9` is preferred.


## Practical Lab / Homework: Causal Masking

In autoregressive models like GPT, a token should not attend to future tokens. This is achieved using a **causal mask** (or look-ahead mask).

**Your Task:**
Create a causal mask for a sequence of length `seq_len = 4` and apply it to the attention mechanism.
A causal mask is a boolean or binary mask where the upper triangle (excluding the diagonal) is set to `True` (or 1) so it gets masked out.


In [2]:
# Create your causal mask for seq_len = 4 here
# Apply it using the scaled_dot_product_attention function above.
# Ensure you do not use stubs, mocks, or TODOs. Provide a fully working implementation.

seq_len_lab = 4
q_lab = np.random.randn(1, seq_len_lab, d_k)
k_lab = np.random.randn(1, seq_len_lab, d_k)
v_lab = np.random.randn(1, seq_len_lab, d_v)

# Write your implementation here:
